<a href="https://colab.research.google.com/github/marques-anderson/risk-segmentation-operational-optimization/blob/main/Risk_Segmentation_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

RISK-BASED CUSTOMER SEGMENTATION & OPERATIONAL OPTIMIZATION

Financial institutions must allocate investigative resources efficiently across customers with varying levels of risk and evaluate whether investigative effort is aligned with the likelihood of escalation.This project analyzes customer risk segmentation, based on a simulated dataset containing various financial attributes. A risk score was then engineered using these variables to approximate financial risk.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
np.random.seed(42)

In [ ]:
n=500

In [ ]:
data = pd.DataFrame({
    'customer_id': range(1, n+1),
    'age': np.random.randint(21, 70, n),
    'income': np.random.randint(30000, 120000, n),
    'credit_score': np.random.randint(300, 850, n),
    'loan_amount': np.random.randint(1000, 50000, n),
    'loan_duration': np.random.randint(6, 60, n),
    'existing_loans': np.random.randint(0, 5, n),
}) ### dataset contains 500 distinct customers, with values randomly generated  between the specified ranges

A risk score was constructed using borrower characteristics, including credit score, loan amount, and existing loans. This score was used to segment customers into low, medium, and high risk groups.


In [ ]:
data['risk_score'] = (
    (700 - data['credit_score']) + ### a credit score of 700 is set as zero risk
    (data['loan_amount'] / 1000) +
    (data['existing_loans'] * 20)  ### loan amounts are given a higher weighting to account for their increase in risk profiles
)

Investigation time is based on the customers risk score, with built in randomness, to simulate real world data.

In [ ]:
data['investigation_time_days'] = (
    data['risk_score'] / 40 + np.random.randint(1, 5, n)
).round(1)


 Risk scores were converted into probabilities so that higher-risk customers were more likely to escalate, but not guaranteed.

In [ ]:
min_score = data['risk_score'].min()
max_score = data['risk_score'].max()

data['escalation_probability'] = 0.1 + 0.8 * (
    (data['risk_score'] - min_score) / (max_score - min_score)
)

In [ ]:
data['random_value'] = np.random.rand(n)
data['escalated'] = (
    data['random_value'] < data['escalation_probability']
).astype(int)

Customers were segmented into three groups using quantile-based binning (qcut), ensuring equal distribution across low, medium, and high risk categories.

In [ ]:
data['segment'] = pd.qcut(
    data['risk_score'],
    3,
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

- High-risk customers exhibited significantly higher escalation rates compared to lower-risk groups.
- Investigation time increased with risk level, indicating that higher-risk cases require more effort.
- Lower-risk customers still consumed investigative resources despite low escalation likelihood, suggesting potential inefficiency.

In [ ]:
data.groupby('segment', observed=False)['investigation_time_days'].mean().plot(kind='bar')
plt.title("Average Investigation Time by Risk Segment")
plt.show()

In [ ]:
data.groupby('segment', observed=False)['escalated'].mean().plot(kind='bar')
plt.title("Escalation Rate by Risk Segment")
plt.show()

A logistic regression model was used to evaluate which variables were most associated with escalation outcomes. The model achieved moderate accuracy and confirmed that borrower risk characteristics were predictive of escalation likelihood.

In [ ]:
X = data[['credit_score', 'loan_amount', 'existing_loans', 'loan_duration']]
y = data['escalated']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
print(accuracy)

In [ ]:
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
})

coefficients = coefficients.sort_values(by='Coefficient', ascending=False)


Most variables aligned with expectations, but one showed an unexpected relationship due to overlap between features in the dataset. Because multiple variables contribute to overall risk, the model must separate their effects, which can sometimes produce less intuitive results. In a real-world setting, this would warrant further investigation.

In [ ]:
coefficients.set_index('Feature')['Coefficient'].plot(kind='bar')

plt.title("Feature Impact on Escalation (Logistic Regression Coefficients)")
plt.xlabel("Feature")
plt.ylabel("Coefficient Value")

plt.axhline(0)
plt.show()